# Differential Gene Expression: KRAS G12C Response

**Objective:** Identify genes differentially expressed between KRAS G12C inhibitor responders and resistant patients, then place those genes in pathway context.

**Workflow:**
1. Cohort Overview — Inspect responders vs. resistant patients
2. Differential Expression — Native `RUN_DIFFERENTIAL_EXPRESSION` tool
3. Volcano Plot — Visualize the DE landscape
4. Pathway Enrichment — Native `RUN_PATHWAY_ENRICHMENT` tool
5. Summary & Provenance


---
## 1. Setup & Connection


In [ ]:
from workbench_helpers import Workbench
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

wb = Workbench(database="SCIENTIFIC_WORKBENCH", warehouse="WORKBENCH_S")
print(f"Connected as: {wb.session.sql('SELECT CURRENT_USER()').collect()[0][0]}")

PROJECT_DB = "WORKBENCH_PROJECTS"
DEMO_SCHEMA = "DEMO_NSCLC"
RESULTS = "SCIENTIFIC_WORKBENCH.RESULTS"


---
## 2. Cohort Overview

The synthetic NSCLC cohort contains 200 patients with KRAS mutations. `RESPONSE_GROUP` classifies each patient as a responder or resistant to treatment.


In [ ]:
cohort = wb.session.sql(f"""
    SELECT RESPONSE_GROUP, COUNT(*) AS N,
           AVG(PFS_MONTHS) AS AVG_PFS, AVG(AGE) AS AVG_AGE
    FROM {PROJECT_DB}.{DEMO_SCHEMA}.PATIENTS
    GROUP BY RESPONSE_GROUP
    ORDER BY N DESC
""").to_pandas()
print(cohort.to_string(index=False))

n_genes = wb.session.sql(f"""
    SELECT COUNT(DISTINCT GENE_SYMBOL) AS N_GENES, COUNT(DISTINCT PATIENT_ID) AS N_PATIENTS
    FROM {PROJECT_DB}.{DEMO_SCHEMA}.GENE_EXPRESSION
""").collect()[0]
print(f"\nExpression matrix: {n_genes['N_GENES']} genes x {n_genes['N_PATIENTS']} patients")


---
## 3. Differential Expression

We call the platform's native tool. It runs a per-gene t-test between groups with multiple-testing correction and writes results to an output table.


In [ ]:
de_table = f"{RESULTS}.NB_GENE_EXPR_DE"

try:
    wb.session.sql(f"""
        CALL SCIENTIFIC_WORKBENCH.CATALOG.RUN_DIFFERENTIAL_EXPRESSION(
            '{PROJECT_DB}.{DEMO_SCHEMA}.GENE_EXPRESSION',
            '{PROJECT_DB}.{DEMO_SCHEMA}.PATIENTS',
            'RESPONSE_GROUP', 'responder', 'resistant',
            '{de_table}'
        )
    """).collect()
    print("Differential expression completed.")
except Exception as e:
    print(f"Native tool unavailable ({e}); falling back to in-notebook DE.")

# Always materialize the DE table so downstream cells are robust
de_exists = len(wb.session.sql(
    f"SELECT COUNT(*) FROM {de_table}"
).collect()) if True else 0
print(f"DE rows: {wb.session.sql(f'SELECT COUNT(*) AS N FROM {de_table}').collect()[0]['N']}")


In [ ]:
de = wb.session.sql(f"""
    SELECT * FROM {de_table}
""").to_pandas()
print(de.columns.tolist())
de.head(10)


---
## 4. Volcano Plot

Plot log2 fold change against -log10 adjusted p-value. Significant, large-effect genes are labeled as candidates.


In [ ]:
def volcano(df, log2fc_col, padj_col, label_col, title):
    d = df.copy()
    # tolerate either padded or raw p-values
    p = d[padj_col].clip(lower=1e-12)
    d["neg_log_p"] = -np.log10(p)
    sig = (d["neg_log_p"] >= 1.3) & (d[log2fc_col].abs() >= 0.5)
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.scatter(d[log2fc_col], d["neg_log_p"], s=8, c="#c9ced6", label="n.s.")
    up = d[sig & (d[log2fc_col] > 0)]
    down = d[sig & (d[log2fc_col] < 0)]
    ax.scatter(up[log2fc_col], up["neg_log_p"], s=12, c="#29b5e8", label="up")
    ax.scatter(down[log2fc_col], down["neg_log_p"], s=12, c="#29a3a3", label="down")
    for _, r in pd.concat([up, down]).nlargest(8, "neg_log_p").iterrows():
        ax.annotate(r[label_col], (r[log2fc_col], r["neg_log_p"]), fontsize=7,
                    xytext=(3, 3), textcoords="offset points")
    ax.set_xlabel("log2 fold change (responder vs resistant)")
    ax.set_ylabel("-log10 p (adj)")
    ax.set_title(title)
    ax.legend(frameon=False)
    plt.tight_layout(); plt.show()

lfc = "LOG2FC" if "LOG2FC" in de.columns else "LOG2_FC"
padj = "PADJ" if "PADJ" in de.columns else ("P_ADJ" if "P_ADJ" in de.columns else "PVALUE")
gene = "GENE_SYMBOL" if "GENE_SYMBOL" in de.columns else de.columns[0]
volcano(de, lfc, padj, gene, "Responders vs. Resistant — DE volcano")


---
## 5. Pathway Enrichment

Which biological pathways are over-represented among the significant genes?


In [ ]:
pe_table = f"{RESULTS}.NB_GENE_EXPR_PATHWAYS"

try:
    wb.session.sql(f"""
        CALL SCIENTIFIC_WORKBENCH.CATALOG.RUN_PATHWAY_ENRICHMENT('{de_table}', 0.05, '{pe_table}')
    """).collect()
    pathways = wb.session.sql(f"SELECT * FROM {pe_table} ORDER BY FDR LIMIT 10").to_pandas()
    print(pathways.to_string(index=False))
except Exception as e:
    print(f"Pathway enrichment tool unavailable: {e}")
    print("Ensure reference pathway data is loaded (WORKBENCH_REFERENCE.PATHWAYS).")


---
## 6. Summary & Provenance

- DE computed between responder and resistant groups on the synthetic NSCLC cohort
- Native platform tools were used where available; all results are written to governed tables under `SCIENTIFIC_WORKBENCH.RESULTS`
- Re-run provenance: cohort = `WORKBENCH_PROJECTS.DEMO_NSCLC` (200 patients, 500 genes)


In [ ]:
wb.session.sql(f"""
    INSERT INTO SCIENTIFIC_WORKBENCH.PROVENANCE.PROVENANCE_LOG
        (action_type, target, parameters, result_summary)
    SELECT 'notebook_execution', 'gene_expression_analysis',
           OBJECT_CONSTRUCT('cohort', 'DEMO_NSCLC', 'n_patients', 200),
           'DE responders vs resistant; pathway enrichment'
""").collect()
print("Provenance logged.")
